# Notebook 3: Model Training & Evaluation
**Input:** `data/` arrays from Notebook 2
**Output:** `data/model_results.json`, `figures/model_comparison.png`


In [ ]:
import pandas as pd, numpy as np, os, json, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (matthews_corrcoef, roc_auc_score,
                              balanced_accuracy_score, confusion_matrix)
from sklearn.metrics import RocCurveDisplay
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

SEED     = 42
DATA_DIR = 'data'
os.makedirs('figures', exist_ok=True)

X_train_fp   = np.load(os.path.join(DATA_DIR,'X_train_ecfp4.npy'))
X_test_fp    = np.load(os.path.join(DATA_DIR,'X_test_ecfp4.npy'))
y_train      = np.load(os.path.join(DATA_DIR,'y_train.npy'))
y_test       = np.load(os.path.join(DATA_DIR,'y_test.npy'))
X_train_desc = pd.read_csv(os.path.join(DATA_DIR,'X_train_desc.csv')).values
X_test_desc  = pd.read_csv(os.path.join(DATA_DIR,'X_test_desc.csv')).values
X_train_comb = np.hstack([X_train_fp, X_train_desc])
X_test_comb  = np.hstack([X_test_fp,  X_test_desc])
print(f'Train: {X_train_comb.shape} | Test: {X_test_comb.shape}')
print(f'Active rate - train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')

In [ ]:
# Model definitions
scale_pos = (y_train==0).sum()/y_train.sum()
models = {
    'RF_ECFP4':     (RandomForestClassifier(n_estimators=500,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=-1), X_train_fp, X_test_fp),
    'RF_Desc':      (RandomForestClassifier(n_estimators=500,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=-1), X_train_desc, X_test_desc),
    'RF_Combined':  (RandomForestClassifier(n_estimators=500,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=-1), X_train_comb, X_test_comb),
    'SVM_ECFP4':    (Pipeline([('sc',StandardScaler()),('svm',SVC(kernel='rbf',class_weight='balanced',probability=True,random_state=SEED))]), X_train_fp, X_test_fp),
    'LR_ECFP4':     (Pipeline([('sc',StandardScaler()),('lr',LogisticRegression(class_weight='balanced',max_iter=1000,random_state=SEED))]), X_train_fp, X_test_fp),
    'XGB_ECFP4':    (XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,scale_pos_weight=scale_pos,eval_metric='logloss',random_state=SEED,n_jobs=-1), X_train_fp, X_test_fp),
    'XGB_Combined': (XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,scale_pos_weight=scale_pos,eval_metric='logloss',random_state=SEED,n_jobs=-1), X_train_comb, X_test_comb),
}
print(f'Models defined: {list(models.keys())}')

In [ ]:
# 5-fold cross-validation
def mcc_scorer(clf,X,y): return matthews_corrcoef(y, clf.predict(X))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}
print('Running 5-fold CV...')
for name,(model,X_tr,_) in models.items():
    scores = cross_validate(model, X_tr, y_train, cv=cv,
                            scoring={'mcc':mcc_scorer,'roc_auc':'roc_auc','balanced_accuracy':'balanced_accuracy'},
                            n_jobs=1, return_train_score=False)
    cv_results[name] = {'cv_mcc_mean':round(scores['test_mcc'].mean(),4),
                        'cv_mcc_std': round(scores['test_mcc'].std(),4),
                        'cv_auc_mean':round(scores['test_roc_auc'].mean(),4),
                        'cv_ba_mean': round(scores['test_balanced_accuracy'].mean(),4)}
    r = cv_results[name]
    print(f'{name:<20} MCC={r["cv_mcc_mean"]:.3f}+/-{r["cv_mcc_std"]:.3f}  AUC={r["cv_auc_mean"]:.3f}')

In [ ]:
# Train on full train set, evaluate scaffold test set
def evaluate(model,X_tr,y_tr,X_te,y_te):
    model.fit(X_tr,y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]
    tn,fp,fn,tp = confusion_matrix(y_te,y_pred).ravel()
    return {'MCC':round(matthews_corrcoef(y_te,y_pred),4),
            'AUC':round(roc_auc_score(y_te,y_prob),4),
            'BA': round(balanced_accuracy_score(y_te,y_pred),4),
            'Sens':round(tp/(tp+fn),4),'Spec':round(tn/(tn+fp),4),
            'y_pred':y_pred,'y_prob':y_prob}

test_results = {}
trained_models = {}
print(f'{"Model":<20} {"MCC":>6} {"AUC":>6} {"BA":>6} {"Sens":>6} {"Spec":>6}')
print('-'*55)
for name,(model,X_tr,X_te) in models.items():
    res = evaluate(model,X_tr,y_train,X_te,y_test)
    test_results[name] = {k:v for k,v in res.items() if k not in ('y_pred','y_prob')}
    trained_models[name] = {'model':model,'y_pred':res['y_pred'],'y_prob':res['y_prob']}
    print(f'{name:<20} {res["MCC"]:>6.3f} {res["AUC"]:>6.3f} {res["BA"]:>6.3f} {res["Sens"]:>6.3f} {res["Spec"]:>6.3f}')

In [ ]:
# ROC curves + MCC comparison plot
fig,axes = plt.subplots(1,2,figsize=(14,5))
colors = ['#2ecc71','#3498db','#e74c3c','#f39c12','#9b59b6','#1abc9c','#e67e22']
for i,(name,res) in enumerate(trained_models.items()):
    RocCurveDisplay.from_predictions(y_test,res['y_prob'],
        name=f"{name} (AUC={test_results[name]['AUC']:.3f})",
        ax=axes[0],color=colors[i%len(colors)],alpha=0.85)
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('ROC Curves - Scaffold Test Set')
axes[0].legend(fontsize=7)
names  = list(test_results.keys())
mccs   = [test_results[n]['MCC'] for n in names]
cv_mcc = [cv_results[n]['cv_mcc_mean'] for n in names]
cv_std = [cv_results[n]['cv_mcc_std']  for n in names]
x = np.arange(len(names)); w = 0.35
axes[1].bar(x-w/2,cv_mcc,w,yerr=cv_std,label='CV MCC',color='steelblue',alpha=0.8,capsize=4,edgecolor='white')
axes[1].bar(x+w/2,mccs,w,label='Test MCC',color='#e74c3c',alpha=0.8,edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(names,rotation=30,ha='right',fontsize=8)
axes[1].set_ylabel('MCC'); axes[1].set_title('CV vs Test MCC')
axes[1].legend(); axes[1].axhline(0,color='gray',linestyle='--',linewidth=0.8)
plt.tight_layout()
plt.savefig('figures/model_comparison.png',dpi=150,bbox_inches='tight')
plt.show()
# Save results
combined = {n:{**{f'cv_{k}':v for k,v in cv_results[n].items()},
               **{f'test_{k}':v for k,v in test_results[n].items()}} for n in test_results}
with open(os.path.join(DATA_DIR,'model_results.json'),'w') as f:
    json.dump(combined, f, indent=2)
best = max(test_results, key=lambda n: test_results[n]['MCC'])
print(f'Best model: {best} | MCC={test_results[best]["MCC"]} | AUC={test_results[best]["AUC"]}')